# 07: Python, C++/CUDA, and JAX together; developing for the stack

**This notebook targets the development stack (LATW `dev` branch).** There is
no Colab path; it needs the development versions installed by
`LISAanalysistools/install.sh`:

```bash
git clone https://github.com/lisa-analysis-tools/lisa-analysis-tools.git LISAanalysistools
bash LISAanalysistools/install.sh
```

Everything below runs on a laptop CPU. GPU (CUDA) and JAX paths are described
and, where a GPU is required to run them, shown as code without executing.

In [1]:
import os

# Threading is pinned to 1 everywhere in this workshop (MPI-only policy;
# OMP-threaded kernels have caused out-of-memory kills on laptops).
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from copy import deepcopy
from lisatools.utils.constants import *

# Slightly larger default text so labels/ticks/titles read clearly in the
# rendered figures.
plt.rcParams.update({"font.size": 12, "axes.titlesize": 13, "axes.labelsize": 12,
                     "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11})


The LISA Analysis Tools stack is a hybrid: pure-Python orchestration on top
of C++/CUDA compute kernels, with a third pure-JAX backend for autodiff. The
same user-facing class can run on NumPy (CPU), CuPy (CUDA), or `jax.numpy`
without changing your code — you pick the backend *once*, at construction. This
notebook shows how that dispatch works, what is compiled where, how to develop
against the editable dev install, and the handful of contributor rules that, if
ignored, break the pipeline in ways that are annoying to debug.

The one architectural idea to take away: **backend selection is centralised in
[`gpubackendtools`](https://github.com/lisa-analysis-tools) and every compute
class dispatches through `self.backend` / `self.backend.xp`.**

### How to read this notebook

- **Casual reader:** read each `##` heading and its **TL;DR**, run the one
  minimal cell.
- **Pipeline developer / contributor:** read the **"Going deeper"**
  subsections and §4 (the house rules) — they are the difference between code
  that composes with the stack and code that fights it. Rules are canonical in
  [`LISAanalysistools/docs/conventions.md`](https://github.com/lisa-analysis-tools/lisa-analysis-tools/blob/dev/docs/conventions.md).

## Backend selection

**TL;DR.** `lisatools.get_backend(name)` returns a `Backend` object exposing
`backend.xp` (either `numpy` or `cupy`) plus the native functions/classes for
that backend; `lisatools.has_backend(name)` reports availability. A compute
class chooses its backend **once, at construction**, via
`force_backend="cpu" | "cuda12x" | "cuda13x" | "jax"` — never as a per-method
keyword — and then dispatches internally through `self.backend.xp`.

In [2]:
import lisatools

be = lisatools.get_backend("cpu")
print("backend:", be.name, "| array module (xp):", be.xp.__name__)

# has_backend reports which plugin wheels are actually installed here.
for name in ("cpu", "cuda11x", "cuda12x"):
    print(f"  has_backend({name!r}) -> {lisatools.has_backend(name)}")

backend: lisatools_cpu | array module (xp): numpy
  has_backend('cpu') -> True
  has_backend('cuda11x') -> False
  has_backend('cuda12x') -> False


In [3]:
from lisatools.utils.parallelbase import LISAToolsParallelModule

# The pattern every compute class follows: subclass LISAToolsParallelModule,
# declare which backends you support, and dispatch through self.backend.xp.
class RMSModule(LISAToolsParallelModule):
    @classmethod
    def supported_backends(cls):
        # names registered in the lisatools_* registry (cuda.../cpu)
        return ["lisatools_" + tag for tag in cls.GPU_RECOMMENDED()]

    def rms(self, arr):
        xp = self.backend.xp          # numpy on cpu, cupy on cuda, jnp on jax
        a = xp.asarray(arr)
        return xp.sqrt(xp.mean(a ** 2))

m = RMSModule(force_backend="cpu")    # <-- backend chosen ONCE, here
print("chosen backend:", m.backend.name, "| xp:", m.backend.xp.__name__)
print("rms([3, 4]) =", float(m.rms([3.0, 4.0])))

chosen backend: lisatools_cpu | xp: numpy
rms([3, 4]) = 3.5355339059327378


### Going deeper: the plugin-wheel model

`lisatools` ships as a **core wheel** plus separate **backend plugin wheels**:
`lisaanalysistools-cpu`, `-cuda11x`, `-cuda12x`, `-cuda13x`. Each plugin
provides a native module (`lisatools_backend_<flavor>.pycppdetector`) that the
core imports lazily and registers under a name — `lisatools_cpu`,
`lisatools_cuda12x`, … — in `gpubackendtools`' global backend registry
(`Globals().backends_manager`). Because the CPU and each CUDA build emit
*distinct* C++ symbol names (the CPU/GPU class-name aliasing rule), the plugins
are side-loadable: `has_backend("cpu")` and `has_backend("cuda12x")` can both be
true in one interpreter, and `get_backend("cuda")` / `"gpu"` alias to the first
available CUDA plugin.

Why "backend at construction, never a method kwarg": one instance = one
backend keeps its arrays, kernels, and dispatcher consistent. Mixing backends
per call creates ambiguous ownership of `xp` arrays, surprise host↔device
copies, and brittle dispatch. Method *names* may carry a backend suffix
(`get_ll_grad_jax`) when the implementation is intrinsically that backend, but
*which* method to call is the caller's choice — not a runtime string.
(See conventions.md, "No backend strings as function kwargs".)

## What's compiled where

**TL;DR.** The native compute lives in each repo's `cutils/` as `.cu`/`.hh`
sources that are compiled **twice**: copied to `.cxx` and built by the C++
compiler for the CPU plugin, and built by `nvcc` for each CUDA plugin. The
shared foundation is `gpubackendtools` (backend registry + reference
cubic-spline C++); on top of it a fourth, pure-JAX backend
(`lisatools.jax`) mirrors the same interfaces in `jax.numpy` for autodiff.

In [4]:
from gpubackendtools.interpolate import CubicSplineInterpolant

# The gpubackendtools foundation: multiple "not-a-knot" cubic splines on one
# backend. A single spline is a 2-D (nsplines=1, length) array.
x = np.linspace(0.0, 2.0 * np.pi, 64)[None, :]
y = np.sin(x)
spline = CubicSplineInterpolant(x, y, force_backend="cpu")

x_new = np.linspace(0.0, 2.0 * np.pi, 400)[None, :]
y_new = spline(x_new)                       # evaluate
dy_new = spline(x_new, derivative=1)        # first derivative (~cos)
print("backend:", spline.backend.name)
print("max |spline - sin| :", float(np.max(np.abs(y_new - np.sin(x_new)))))
print("max |deriv  - cos| :", float(np.max(np.abs(dy_new - np.cos(x_new)))))

backend: gbt_cpu
max |spline - sin| : 2.8055839840918084e-07
max |deriv  - cos| : 1.7185169945976853e-05


### Going deeper: the native layer (`cutils/`)

- Sources like `lisatools/cutils/Detector.cu` (LISA orbits / detector geometry)
  are written to be valid as **both** C++ and CUDA. At build time the `.cu` is
  copied to `Detector.cxx` and compiled by the C++ compiler for the CPU plugin,
  while the same `.cu` is compiled by `nvcc` for each CUDA plugin — so a change
  to `Detector.cu` rebuilds both targets. `gpubackendtools`' `Interpolate.cu`
  (the spline above) follows the identical `.cu → .cxx` pattern.
- **Single-registrant rule (one sentence):** only LAT registers the shared
  nanobind wrapper classes (`OrbitsWrap`, `TDIConfigWrap`, …) with the type
  system — downstream wheels (GBGPU, BBHx, FEW) recompile LAT/GBT headers in
  place but must *not* re-register those wrappers, a rule enforced at compile
  time by the `LISATOOLS_IS_WRAPPER_OWNER` macro. GBT owns `CubicSplineWrap`.
- Downstream wheels consume upstream C++/CUDA by **recompiling against the
  headers**, not linking the compiled archive, and prefer stable-layout POD
  `*View` structs (`OrbitsView`) as the cross-wheel interface (conventions.md,
  "Cross-wheel C++/CUDA sharing").

### Going deeper: the pure-JAX mirror (`lisatools.jax`)

`lisatools.jax` is a fourth backend that mirrors `lisatools.cutils` but uses
`jax.numpy` and autograd-friendly Python — no compiled plugin wheel, gated only
on `import jax`. It is selected the same way as the others
(`force_backend="jax"` / `get_backend("jax")`) and resolves to a
`LISAToolsJaxBackend`. Its purpose is *gradients*: with the JAX response +
waveform path you can `jax.grad` straight through a LISA likelihood and feed the
gradient to a NUTS/HMC sampler.

The mechanic is just standard JAX (this runs in a second on CPU):

In [5]:
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# jax.grad of a trivial scalar "likelihood" -- proves the autodiff path is live.
def toy_loglike(z):
    return -jnp.sum(jnp.sin(z) ** 2)

grad_toy = jax.grad(toy_loglike)
z0 = jnp.array([0.3, 1.1, 2.0])
print("grad:", np.asarray(grad_toy(z0)))
print("check (-2 sin z cos z):", np.asarray(-2 * jnp.sin(z0) * jnp.cos(z0)))

grad: [-0.56464247 -0.8084964   0.7568025 ]
check (-2 sin z cos z): [-0.56464247 -0.8084964   0.7568025 ]


A *real* GB likelihood gradient looks like the sketch below — a GB
TDI-on-the-fly template built on the `jax` backend, wrapped in an
`AnalysisContainer`, differentiated with `jax.grad`. The full runnable version
is `LISAanalysistools/examples/gb_jax_likelihood_grad.py`. **It is not executed
here:** the eager TDI-on-the-fly graph takes minutes per gradient on a CPU, and
JAX gradients through the response are *GPU-recommended*.

```python
import jax, jax.numpy as jnp
from lisatools.analysiscontainer import AnalysisContainer
from lisatools.detector import EqualArmlengthOrbits
from lisatools.jax.response import TDIConfigWrapJAX
from lisatools.response.tdiconfig import TDIConfig
from gbgpu.jax.tdi_on_the_fly import gb_run_wave_tdi
jax.config.update("jax_enable_x64", True)

# ... build orbits_jax / tdi_jax on the jax backend (see the example) ...

def gb_template_td(theta, t_arr, t_ref, orbits_jax, tdi_jax):
    M, *_ = gb_run_wave_tdi(theta[None, :], t_arr[None, :], t_ref,
                            orbits_jax, tdi_jax)
    return jnp.real(M[0])                       # (nchannels, N)

def loglike(theta):                             # delegates to AnalysisContainer
    h_fd = jnp.fft.rfft(gb_template_td(theta, ...), axis=-1) * dt
    return analysis.template_likelihood(h_fd)   # log L = -0.5 <d-h|d-h>

grad_loglike = jax.grad(loglike)                # <-- autodiff through the response
```

The JAX backend may diverge *internally* from the C++ kernels (it uses
`jax.lax.scan` / `vmap` / static-shape masking rather than shared-memory
tiling) but must match the C++ **inner-product** outputs to `reldiff ≲ 1e-12`
(conventions.md, "Backend implementation hierarchy").

## The dev workflow

**TL;DR.** `install.sh` lays the stack out as sibling clones on their
development branches and editable-installs each one with a pinned constraint
set. After that, **pure-Python edits are live immediately** (editable install),
but **edits to `.cu`/`.hh`/`.cxx` native code require a rebuild** — and a stale
`.so` from a half-finished build is the classic trap.

In [6]:
import lisatools

# A ".dev" local version string is the fingerprint of an editable dev build.
print("lisatools:", lisatools.__version__)
print("editable install:", lisatools._is_editable)

lisatools: 1.2.8.post1.dev750+g636b5677d.d20260706
editable install: True


### Going deeper: `install.sh` anatomy

`LISAanalysistools/install.sh` is the single entry point. It:

1. **Clones siblings side by side.** From your `lisa-analysis-tools/` clone it
   clones `GPUBackendTools`, `Eryn`, `BBHx`, `GBGPU`, `LATW`, and (optionally)
   `FastEMRIWaveforms` into the *same parent directory*, reusing existing
   clones. This side-by-side layout is why cross-repo header includes and the
   umbrella docs resolve.
2. **Checks out development branches** per package — GBT `spline`, Eryn `dev`,
   LAT `dev`, BBHx `dev`, GBGPU `dev`, FEW `gpu_backend`, LATW `dev` — and
   fast-forward-pulls each (aborting if a branch has diverged, so you never
   build on a silently stale tree).
3. **Pins the build with `PIP_CONSTRAINT`.** It exports
   `PIP_CONSTRAINT=constraints/sprint.txt` (the LISA-Analysis-Tools-wide
   nanobind/pybind11 pin) before every `pip install`, so all wheels share one
   ABI.
4. **Editable, no build isolation.** Each package is installed with
   `pip install --no-build-isolation -e .`, in dependency order (GBT → Eryn →
   LAT → BBHx/GBGPU → FEW), so downstream builds see the already-installed
   upstream headers.
5. **Finds LAPACKE through GBT.** GBT's detector is the single source of the
   `GBT_LAPACKE_*` options; the same flag set is forwarded to every compiled
   package (LAT, BBHx, GBGPU, FEW).

### Going deeper: the edit → rebuild loop (and the stale-`.so` trap)

- **Pure-Python edits are live.** Because everything is an editable install,
  changing a `.py` under any `src/…` takes effect on the next `import` (or
  kernel restart) — no rebuild.
- **Native edits need a rebuild.** Touch a `.cu`/`.hh`/`.cxx` and you must
  recompile the affected package. Re-run its editable install and **capture the
  build log to a file** — never pipe a long native build through `| tail`:

  ```bash
  cd GPUBackendTools
  PIP_CONSTRAINT=../lisa-analysis-tools/constraints/sprint.txt \
      pip install --no-build-isolation -e . > /tmp/gbt_build.log 2>&1
  echo "exit=$?"; tail -n 40 /tmp/gbt_build.log
  ```

- **The stale-`.so` trap.** `pip | tail` (and interrupted builds) can leave the
  *old* compiled `.so` in place while the command reports success from the
  truncated pipe — so you keep importing the previous binary and your C++ change
  "does nothing". Always redirect to a file, check the real exit code, and grep
  the log for the compile line for the file you changed. When a cross-wheel cast
  fails, the first hypothesis is a stale downstream built against an older
  `LISATOOLS_HEADER_ABI_VERSION` — rebuild the downstream before deeper
  debugging.
- **Repo/branch map.** The package ↔ dev-branch ↔ pip-name table lives in
  notebook 00 (§"The ecosystem at a glance") and
  `LISAanalysistools/docs/architecture-map.md`.

## Contributor house rules that bite

**TL;DR.** Four rules catch nearly everyone. They are enforced across *all*
stack repos and documented in
[`docs/conventions.md`](https://github.com/lisa-analysis-tools/lisa-analysis-tools/blob/dev/docs/conventions.md);
ignoring them produces bugs that are slow to trace (silent stale binaries,
`cannot pickle 'module' object`, OOM-killed laptops).

In [7]:
import pickle
from lisatools.utils.parallelbase import LISAToolsParallelModule

# Rule 2 in action: a settings-like object must survive deepcopy + pickle.
class Settings(LISAToolsParallelModule):
    @classmethod
    def supported_backends(cls):
        return ["lisatools_" + tag for tag in cls.GPU_RECOMMENDED()]
    def __init__(self, center_freq=8e-3, force_backend=None):
        super().__init__(force_backend=force_backend)
        self.center_freq = center_freq

s = Settings(force_backend="cpu")
print("xp stored as an attribute? ", "xp" in s.__dict__, " (must be False)")

# GlobalFitSetup (formerly CurrentInfoGlobalFit) deepcopies the whole settings tree; MPI workers pickle it.
s2 = pickle.loads(pickle.dumps(deepcopy(s)))
print("survives deepcopy+pickle ->", s2.backend.name, "| xp:", s2.xp.__name__,
      "| center_freq:", s2.center_freq)

xp stored as an attribute?  False  (must be False)
survives deepcopy+pickle -> lisatools_cpu | xp: numpy | center_freq: 0.008


### Going deeper: the four rules

1. **Use `force_backend`, not `use_gpu`.** Backend choice is made once, at
   construction, with `force_backend="cpu"` / `"cuda12x"` / `"jax"`. There is no
   `use_gpu` / `backend` / `use_jax` keyword on methods — dispatch internally
   through `self.backend`. (The retired `use_gpu`-era keyword is gone from
   user-facing classes; do not reintroduce it.)

2. **Deepcopy / pickle safety.** Objects routinely travel through
   `copy.deepcopy` (the settings tree) and `pickle` (MPI workers,
   multiprocessing, cached states). Anything transitively holding a raw Python
   **module** dies with `TypeError: cannot pickle 'module' object`. So **never
   store an array module** (`self.xp = cp`) — expose `xp` as a property derived
   from a flag or from `self.backend` (which pickles as a registry-name
   reference and deepcopies to itself). The cell above proves the round-trip.
   `__getattr__` delegators must guard their storage attribute and dunders so
   `copy`/`pickle` probing does not recurse.

3. **`OMP_NUM_THREADS=1` / MPI-only — and *why*.** Threading is owned at the
   *run* level (MPI ranks), not inside kernels: there is **no nested OpenMP in
   compute kernels** anywhere in the stack. On a laptop, OMP-threaded kernels
   have oversubscribed cores and driven memory use up until the OS **SIGKILLed
   the process** — so every notebook here pins `OMP_NUM_THREADS=1` (plus the
   OPENBLAS/MKL/VECLIB/NUMEXPR twins) in cell 3, and you scale out with
   `mpiexec` ranks instead of threads. Fix a slow CPU kernel algorithmically or
   move it to the GPU — never with `#pragma omp` inside a kernel.

4. **No new global-fit settings files.** Run configurations are **installed
   stock classes**, not settings files you copy and edit:

   ```python
   from lisatools.globalfit.stock import erebor
   fit = erebor.gb_no_fg(nwalkers=4)     # a StockGlobalFit subclass
   fit.gb.center_freq = 8e-3             # plain attribute access
   fit.recipe.pop_move("rj_refit")       # named move stacks per stage
   fit.build(); fit.run()                # heavy work only on command
   ```

   A new variant is a `StockGlobalFit` subclass under
   `lisatools/globalfit/stock/erebor/variants/` with documented dataclass knobs
   — nothing heavy in `__init__`, and the pre-build fit must pickle/deepcopy
   (rule 2). Notebooks 02 and 08 use this API throughout.

All four are canonical in
[`docs/conventions.md`](https://github.com/lisa-analysis-tools/lisa-analysis-tools/blob/dev/docs/conventions.md)
and [`LISAanalysistools/CLAUDE.md`](https://github.com/lisa-analysis-tools/lisa-analysis-tools/blob/dev/CLAUDE.md).